# C1 Session 3 — Metrics: Accuracy, Precision, Recall, F1, and Class Imbalance

*Machine Learning Fundamentals, session 3 of 3 (~80 min). Prerequisites:
F1-scientific-python and Sessions 1–2.*

Sessions 1–2 judged every rule by accuracy. This session shows why that
single number is not enough: metrics that separate the *kinds* of mistakes,
the macro average for multi-class tasks, and the imbalanced datasets where
accuracy actively lies. Answers to all checkpoints are at the end of this
notebook.

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

## 1. The Four Counts and the Confusion Matrix

**Motivation.** Accuracy is a blender — it purees *what kind* of mistakes a
rule makes into a single smoothie. A medical screen that misses sick patients
and one that cries wolf at healthy patients can have the *same* accuracy
while being wrong in completely different ways. We need numbers that keep the
mistake types apart.

**The four counts.** For a two-class task, call one class **positive** (label
`1`, usually the class we are hunting for) and the other **negative** (label
`0`). Every prediction lands in exactly one of four buckets:

|                    | predicted 1        | predicted 0        |
|--------------------|--------------------|--------------------|
| **actually 1**     | TP (true positive) | FN (false negative) |
| **actually 0**     | FP (false positive)| TN (true negative)  |

This 2×2 table of counts is the **confusion matrix** — it shows exactly
where the rule gets confused.

**Worked example.** Twelve paired labels, counted with boolean masks
(elementwise comparisons and `np.sum`, as covered in F1-scientific-python):

In [ ]:
y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0])

TP = np.sum((y_true == 1) & (y_pred == 1))
FN = np.sum((y_true == 1) & (y_pred == 0))
FP = np.sum((y_true == 0) & (y_pred == 1))
TN = np.sum((y_true == 0) & (y_pred == 0))
print("TP =", TP, " FN =", FN, " FP =", FP, " TN =", TN)

Check the counts by eye: positions 0, 3, 6, 8 are hits on true positives
(TP = 4); positions 2 and 11 are missed positives (FN = 2); position 5 is a
false alarm (FP = 1); the remaining five are correctly ignored negatives
(TN = 5).

### Checkpoint 1

1. An example has `y_true = 1, y_pred = 0` — which bucket? And
   `y_true = 0, y_pred = 1`?
2. Count all four buckets by eye for `y_true = [1, 0, 0, 1]`,
   `y_pred = [1, 1, 0, 0]`.

## 2. Accuracy, Precision, Recall, and F1

Each metric answers a different question about the four counts:

- **Accuracy** — *what fraction of all answers were right?*
  $\text{accuracy} = \dfrac{TP + TN}{TP + TN + FP + FN}$
- **Precision** — *when the rule said "positive", how often was it right?*
  $\text{precision} = \dfrac{TP}{TP + FP}$ — penalizes false alarms.
- **Recall** — *of the actual positives, what fraction did the rule find?*
  $\text{recall} = \dfrac{TP}{TP + FN}$ — penalizes misses.
- **F1** — precision and recall pull in opposite directions (flag everything
  → perfect recall, awful precision; flag almost nothing you're unsure of →
  precision rises while recall collapses), so we combine them:
  $F_1 = \dfrac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$.
  This is the *harmonic* mean, and unlike the ordinary average it stays low
  unless **both** are decent: precision 1.0 with recall 0.1 averages to 0.55
  but has F1 ≈ 0.18. You cannot buy a good F1 with one great number and one
  terrible one.

**The arithmetic, worked.** With Section 1's counts TP = 4, FN = 2, FP = 1,
TN = 5: accuracy = 9/12 = 0.75; precision = 4/5 = 0.8; recall = 4/6 ≈ 0.667;
F1 = 2 · 0.8 · 0.667 / (0.8 + 0.667) ≈ 0.727. Verified with NumPy:

In [ ]:
acc  = (TP + TN) / (TP + TN + FP + FN)
prec = TP / (TP + FP)
rec  = TP / (TP + FN)
f1   = 2 * prec * rec / (prec + rec)
print(f"accuracy  = {acc:.3f}")
print(f"precision = {prec:.3f}")
print(f"recall    = {rec:.3f}")
print(f"F1        = {f1:.3f}")

### Checkpoint 2

1. A rule produces TP = 8, FP = 2, FN = 4, TN = 16. Compute accuracy,
   precision, recall, and F1 by hand.
2. Two failure styles: rule A misses many real positives but almost never
   raises a false alarm; rule B catches nearly every positive but flags many
   negatives too. Which metric is low for A, and which for B?

## 3. Macro-F1 for More Than Two Classes

**Motivation.** Precision, recall, and F1 are defined for a *positive class*
— but many tasks have three or more classes (cat / dog / rabbit). The fix is
**one-vs-rest**: score each class as if it were the positive class and every
other class were negative, then average. The average of the per-class F1
scores is the **macro-F1**. "Macro" signals that every class counts equally
in the average — a rare class weighs as much as a common one.

**Worked example.** Fifteen animal photos, three classes
(`0` = cat, `1` = dog, `2` = rabbit). First the 3×3 confusion table
(rows = actual class, columns = predicted class):

In [ ]:
y_true3 = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2])
y_pred3 = np.array([0, 0, 0, 0, 1, 1, 1, 1, 0, 2, 2, 2, 2, 2, 1])

K = 3
C = np.zeros((K, K), dtype=int)
for actual, predicted in zip(y_true3, y_pred3):
    C[actual, predicted] += 1
print(C)

Row 0 reads: of the 5 actual cats, 4 were predicted cat and 1 dog. The
diagonal holds the correct predictions. From this table, everything follows
with axis-wise sums (`axis=0` down columns, `axis=1` across rows, as covered
in F1-scientific-python):

- class *k*'s **precision** = `C[k, k]` / (column-*k* sum) — of everything
  *predicted* class *k*, the fraction truly class *k*;
- class *k*'s **recall** = `C[k, k]` / (row-*k* sum) — of everything
  *actually* class *k*, the fraction found.

In [ ]:
diag = np.diag(C).astype(float)
prec_per_class = diag / C.sum(axis=0)     # column sums: all predicted-k
rec_per_class  = diag / C.sum(axis=1)     # row sums: all actual-k
f1_per_class   = 2 * prec_per_class * rec_per_class / (prec_per_class + rec_per_class)

for k, name in enumerate(["cat", "dog", "rabbit"]):
    print(f"{name:>6}: precision={prec_per_class[k]:.3f}  "
          f"recall={rec_per_class[k]:.3f}  F1={f1_per_class[k]:.3f}")

macro_f1 = f1_per_class.mean()
print(f"macro-F1 = {macro_f1:.3f}")

**The arithmetic, worked, for the dog class** (k = 1): column-1 sum is
1 + 3 + 1 = 5 predicted dogs, of which 3 truly are → precision = 3/5 = 0.6.
Row-1 sum is 5 actual dogs, 3 found → recall = 3/5 = 0.6. So dog-F1 = 0.6.
Cats and rabbits each score F1 = 0.8, so
macro-F1 = (0.8 + 0.6 + 0.8) / 3 ≈ 0.733. The weak dog class drags the
average down — exactly the sensitivity to per-class performance we want.

### Checkpoint 3

1. In a 3-class confusion table, what does the sum of row *k* count, and
   what does the sum of column *k* count?
2. A rule scores F1 = 0.9 on two frequent classes and F1 = 0.1 on one rare
   class. What is its macro-F1, and what does the value warn you about?

## 4. When Accuracy Lies: Class Imbalance

**Motivation.** Time to see why we bothered. A dataset has **class
imbalance** when one class vastly outnumbers the other — fraud among
transactions, defects among parts, one rare disease among healthy patients.
Under imbalance, plain accuracy becomes actively misleading.

**Worked example: the 99%-negative demo.** 1000 patient screenings; only 10
patients actually have the condition (label `1`). Consider the laziest
possible "classifier": **predict negative for everyone.**

In [ ]:
y_true = np.zeros(1000, dtype=int)
y_true[:10] = 1                       # ten actual positives

pred_silent = np.zeros(1000, dtype=int)   # "nobody has it"

TP = np.sum((y_true == 1) & (pred_silent == 1))
FN = np.sum((y_true == 1) & (pred_silent == 0))
FP = np.sum((y_true == 0) & (pred_silent == 1))
TN = np.sum((y_true == 0) & (pred_silent == 0))

acc_silent = (TP + TN) / 1000
rec_silent = TP / (TP + FN)
# The silent rule never predicts positive, so TP + FP = 0 and precision's
# fraction has denominator zero. Convention: a rule that finds no positives
# at all gets F1 = 0.
f1_silent = 0.0
print("silent rule: accuracy =", acc_silent, " recall =", rec_silent, " F1 =", f1_silent)

**99% accurate** — and it never detects a single sick patient. The accuracy
number is not lying about the fraction correct; it is lying about
*usefulness*, because with 990 negatives you can be right 99% of the time
while ignoring the entire point of the screening. Recall (0.0) and F1 (0)
expose the fraud instantly.

Now a genuine (imperfect) detector: it catches 7 of the 10 patients, at the
price of 15 false alarms:

In [ ]:
pred_detector = np.zeros(1000, dtype=int)
pred_detector[:7] = 1                 # finds 7 of the 10 actual positives
pred_detector[10:25] = 1              # 15 false alarms among the negatives

TP = np.sum((y_true == 1) & (pred_detector == 1))
FN = np.sum((y_true == 1) & (pred_detector == 0))
FP = np.sum((y_true == 0) & (pred_detector == 1))
TN = np.sum((y_true == 0) & (pred_detector == 0))

acc_det  = (TP + TN) / 1000
prec_det = TP / (TP + FP)
rec_det  = TP / (TP + FN)
f1_det   = 2 * prec_det * rec_det / (prec_det + rec_det)
print(f"detector: accuracy = {acc_det:.3f}  precision = {prec_det:.3f}  "
      f"recall = {rec_det:.3f}  F1 = {f1_det:.3f}")

By accuracy, the *silent rule beats the detector* (0.990 vs 0.982). By every
metric that respects the rare class, the detector wins in a landslide: it
finds 70% of the sick patients instead of 0%. Moral:

- **Under class imbalance, never judge by accuracy alone.** Compare against
  the always-majority baseline first.
- **Recall answers "what fraction of the rare cases did we catch?"** —
  usually the question that matters when missing a case is costly.
- **Precision answers "how many alarms were real?"** — the cost of
  follow-ups.
- **F1 keeps both honest** in a single number.

### Checkpoint 4

1. In a stream of transactions, 1 in 500 is fraudulent. A rule predicts
   "not fraud" every time. What are its accuracy and recall?
2. Your team celebrates a screening rule with 97% accuracy on data that is
   98% negative. What single comparison should instantly cool the
   celebration?

## 5. Common Pitfalls (Metric Edition)

**Pitfall 1: swapping precision and recall.** The two formulas differ only
in one letter of the denominator, and mixing them up flips the meaning of a
report. Broken:

In [ ]:
pit_true = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
pit_pred = np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0])

TPp = np.sum((pit_true == 1) & (pit_pred == 1))
FPp = np.sum((pit_true == 0) & (pit_pred == 1))
FNp = np.sum((pit_true == 1) & (pit_pred == 0))

# BROKEN: this is NOT precision -- dividing by TP + FN gives recall.
broken_precision = TPp / (TPp + FNp)
print("claimed 'precision':", broken_precision, " (actually recall!)")

# FIX: precision judges the rule's positive CALLS (TP + FP);
#      recall judges coverage of the REAL positives (TP + FN).
precision = TPp / (TPp + FPp)
recall    = TPp / (TPp + FNp)
print("precision:", round(precision, 3), "  recall:", recall)

Memory hook: **p**recision = of my **p**ositive calls, how many were right;
**r**ecall = of the **r**eal positives, how many did I find.

**Pitfall 2: row/column mix-up in a confusion table.** With rows = actual
and columns = predicted, class *k*'s precision divides by the **column** sum
and its recall by the **row** sum. Using `axis=1` for both silently computes
recall twice. Broken:

In [ ]:
Cp = np.array([[8, 2, 0],
               [1, 6, 3],
               [0, 1, 9]])
diag_p = np.diag(Cp).astype(float)

# BROKEN: row sums (axis=1) are the ACTUAL-class counts -- this is recall.
broken_prec = diag_p / Cp.sum(axis=1)
print("claimed 'precision':", broken_prec)

# FIX: precision needs the predicted-class counts -- column sums (axis=0).
prec_fixed = diag_p / Cp.sum(axis=0)
print("actual  precision :", prec_fixed)

The two versions disagree on every class (class 0: 0.8 claimed vs 0.889
actual) and nothing crashes — the shapes match perfectly. Restate to
yourself which axis is which before trusting the numbers.

**Pitfall 3: "accuracy is fine, so the classifier is fine" — pooling vs.
the macro average.** Overall accuracy pools all examples into one fraction,
so frequent classes dominate it; macro-F1 weights classes equally. Broken
reading:

In [ ]:
Ci = np.array([[80, 3, 2],    # class 0: 85 examples (frequent)
               [ 3, 6, 1],    # class 1: 10 examples
               [ 2, 2, 1]])   # class 2:  5 examples (rare)

overall_accuracy = np.trace(Ci) / Ci.sum()   # trace = diagonal sum
print("overall accuracy:", overall_accuracy)   # looks decent...

# FIX: also compute the equal-weight (macro) view before judging.
d = np.diag(Ci).astype(float)
prec_c = d / Ci.sum(axis=0)
rec_c  = d / Ci.sum(axis=1)
f1_c   = 2 * prec_c * rec_c / (prec_c + rec_c)
print("per-class F1:", np.round(f1_c, 3))
print("macro-F1:    ", round(f1_c.mean(), 3))

Accuracy 0.87 hides that the rare class is nearly always wrong (its F1 ≈
0.22); macro-F1 ≈ 0.58 refuses to hide it. Neither number is "the truth" —
they weight classes differently — but reporting only the pooled one on
imbalanced data is how bad classifiers get deployed.

### Checkpoint 5

1. Spot the bug: `recall = TP / (TP + FP)`. What quantity does this actually
   compute, and what is the correct denominator for recall?
2. In a confusion table with rows = actual, which axis sum feeds precision
   and which feeds recall?

## 6. Exam Connections and Going Deeper

Round 1 rewards fluency with these metrics twice over (see
`reference/analysis.md`, topic-distribution table): the "ML concepts"
cluster's five-option MC items include metric computations with
normal-form numeric answers, and the applied tabular problem — the exam's
single biggest arc — scores its submissions by **macro-F1**, so
understanding exactly what that number rewards is worth points. Two worked
items in the exam's register:

---

**Worked exam-style example 1 (multiple choice).**
*Reasoning is required. No coding is needed.*

A binary classifier is evaluated on 50 examples: TP = 12, FP = 8, FN = 3,
TN = 27. Written as a fraction m/n in lowest terms (gcd(m, n) = 1, n > 0),
its F1 score is:

- **A.** 3/5 &nbsp; **B.** 4/5 &nbsp; **C.** 24/35 &nbsp; **D.** 7/10 &nbsp; **E.** 12/25

*Worked solution.* Step 1 — precision: TP/(TP+FP) = 12/20 = **3/5**.
Step 2 — recall: TP/(TP+FN) = 12/15 = **4/5**.
Step 3 — F1 is the harmonic mean:
F1 = 2 · (3/5)(4/5) / (3/5 + 4/5) = 2 · (12/25) / (7/5) = (24/25) · (5/7)
= 120/175 = **24/35**.
Step 4 — normal form: gcd(24, 35) = 1, so m/n = 24/35 exactly as required →
**C**. Distractor anatomy: A is the precision, B is the recall, D is the
*arithmetic* mean of the two (the classic wrong average), E is their
product. Verified numerically below.

In [ ]:
TPe, FPe, FNe, TNe = 12, 8, 3, 27
p_ex = TPe / (TPe + FPe)
r_ex = TPe / (TPe + FNe)
f1_ex = 2 * p_ex * r_ex / (p_ex + r_ex)
print("precision:", p_ex, " recall:", r_ex, " F1:", f1_ex, "= 24/35 =", 24 / 35)

---

**Worked exam-style example 2 (constrained coding).**
*Coding is required. Reasoning may be shown as comments.*

Write code to implement

```python
def metric_report(y_true, y_pred):
    ...
```

taking two equal-length NumPy integer arrays of 0/1 labels and returning
the tuple `(accuracy, precision, recall, f1)` as four floats. **Banned
inside the function: Python `for` and `while` loops and list
comprehensions. Any use of a banned construct scores zero points for this
part.** You may assume at least one positive call and at least one positive
example.

*Worked solution.* Step 1 — the contract: two array arguments in, a 4-tuple
of floats out, exact name `metric_report`. Step 2 — the loop ban points
straight at boolean masks: each confusion count is one elementwise
comparison pair combined with `&` and summed. Step 3 — the four metrics are
pure arithmetic on the counts. Step 4 — verify on data with known counts
(below we reuse example 1's counts, so the output must reproduce 24/35):

In [ ]:
def metric_report(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    accuracy  = (TP + TN) / y_true.size
    precision = TP / (TP + FP)
    recall    = TP / (TP + FN)
    f1        = 2 * precision * recall / (precision + recall)
    return float(accuracy), float(precision), float(recall), float(f1)

# Arrays engineered to have TP=12, FP=8, FN=3, TN=27 (order is irrelevant
# to the counts):
exam_true = np.concatenate([np.ones(15, dtype=int), np.zeros(35, dtype=int)])
exam_pred = np.concatenate([np.ones(12, dtype=int), np.zeros(3, dtype=int),
                            np.ones(8, dtype=int), np.zeros(27, dtype=int)])
print(metric_report(exam_true, exam_pred))
print("expected F1:", 24 / 35)

**Going deeper (optional).** In **C4-classical-ml-practice** these metrics
score real classifiers on realistic tabular data, macro-F1 included, using
a standard toolkit. **F5-probability** explains how much a metric measured
on a finite test set can be trusted — the formal footing for why we prefer
larger test sets. Neither is needed for this unit's practice.

### Checkpoint 6

1. Why does the "fraction m/n in lowest terms, n > 0" constraint make a
   numeric answer uniquely decodable? What ambiguity would remain without
   it?
2. A contestant's `metric_report` returns exactly the right numbers but
   builds the counts with a `for` loop. Under the exam's register it scores
   zero. What skill is the ban actually testing?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. `y_true = 1, y_pred = 0` is a **false negative** (a missed positive);
   `y_true = 0, y_pred = 1` is a **false positive** (a false alarm).
2. Position by position: (1,1) → TP; (0,1) → FP; (0,0) → TN; (1,0) → FN.
   So TP = 1, FP = 1, TN = 1, FN = 1.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. accuracy = (8+16)/30 = 0.8; precision = 8/10 = 0.8; recall = 8/12 ≈
   0.667; F1 = 2 · 0.8 · 0.667 / 1.467 ≈ 0.727.
2. Rule A's misses drive **recall** down; rule B's false alarms drive
   **precision** down.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Row *k* sums to the number of examples *actually* in class *k*; column
   *k* sums to the number of examples *predicted* as class *k*.
2. macro-F1 = (0.9 + 0.9 + 0.1)/3 ≈ 0.633. The value warns that at least
   one class is being handled badly even though the frequent classes look
   fine — equal weighting lets the rare class pull the average down.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Accuracy = 499/500 = 0.998; recall = 0 — it catches no fraud at all.
2. Compare with the always-negative baseline: predicting "negative" for
   everyone already scores 98% accuracy, so 97% is *below* the do-nothing
   baseline. Ask for recall, precision, and F1.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. `TP / (TP + FP)` is **precision** (it judges the positive calls). Recall
   divides by the actual positives: `TP / (TP + FN)`.
2. Column sums (`axis=0`, the predicted-class totals) feed precision; row
   sums (`axis=1`, the actual-class totals) feed recall.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. A numeric value has many equivalent written forms (24/35, 48/70,
   0.6857…). Requiring lowest terms with a positive denominator pins down a
   single (m, n) pair, so grading can match one exact answer with no
   judgment calls.
2. Working at the array level: expressing a computation as elementwise
   operations, masks, and aggregations instead of element-by-element loops.
   The exam bans the loop precisely so that a correct-looking output cannot
   substitute for demonstrating that skill.

</details>